In [1]:
import os
os.getcwd()

'/home/sachin/Desktop/RAGDemo/notebook'

#### Data Ingestion

In [2]:
from dotenv import load_dotenv
load_dotenv()

import langsmith

In [3]:
from langchain_core.documents import Document

In [4]:
doc = Document(
    page_content="this is the main content for the rag application",
    metadata ={
        "source":"example.txt",
        "pages" : 1,
        "author" : "Sachin",
        "date_created" : "2025-11-25"
    }
)
doc

Document(metadata={'source': 'example.txt', 'pages': 1, 'author': 'Sachin', 'date_created': '2025-11-25'}, page_content='this is the main content for the rag application')

In [5]:
import os
os.makedirs("../data/text_files", exist_ok = True)

In [6]:
sample_texts={
    "../data/text_files/python_intro.txt":"""Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.""",
    
    "../data/text_files/machine_learning.txt": """Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems
    
    
    """

}

for filepath,content in sample_texts.items():
    with open(filepath,'w',encoding="utf-8") as f:
        f.write(content)

print("✅ Sample text files created!")

✅ Sample text files created!


In [7]:
# TextLoader
# from langchain.document_loaders import TextLoader
from langchain_community.document_loaders import TextLoader

/home/sachin/Desktop/RAGDemo/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
loader = TextLoader("../data/text_files/python_intro.txt", encoding="utf-8")
document = loader.load()
print(document)

[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.')]


In [9]:
from langchain_community.document_loaders import DirectoryLoader
dir_loader = DirectoryLoader(
    "../data/text_files",
    glob = "**/*.txt",
    loader_cls = TextLoader,
    loader_kwargs={'encoding':'utf-8'},
    show_progress=False,
)

documents = dir_loader.load()
print(documents)

[Document(metadata={'source': '../data/text_files/machine_learning.txt'}, page_content='Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed. It focuses on developing computer programs\nthat can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Learning: Finding patterns in unlabeled data\n3. Reinforcement Learning: Learning through rewards and penalties\n\nApplications include image recognition, speech processing, and recommendation systems\n\n\n    '), Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\npr

In [10]:
from langchain_community.document_loaders import PyMuPDFLoader,PyPDFLoader
dir_loader = DirectoryLoader(
    "../data/pdf",
    glob = "**/*.pdf",
    loader_cls = PyMuPDFLoader,
    show_progress=False,
)

documents = dir_loader.load()
print(documents)

[Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2025-07-28T16:15:39+05:30', 'source': '../data/pdf/Module 3.pdf', 'file_path': '../data/pdf/Module 3.pdf', 'total_pages': 19, 'format': 'PDF 1.7', 'title': '', 'author': 'Shahedhadeennisa Shaik', 'subject': '', 'keywords': '', 'moddate': '2025-07-28T16:15:39+05:30', 'trapped': '', 'modDate': "D:20250728161539+05'30'", 'creationDate': "D:20250728161539+05'30'", 'page': 0}, page_content='DAYANANDA SAGAR COLLEGE OF ENGINEERING \n         An Autonomous Institute Affiliated to VTU, Belagavi Approved by AICTE; ISO 9001:2015 \n        Certified Accredited by National Assessment Accreditation Council (NAAC) with ‘A’ grade \nShavige Malleshwara Hills, Kumaraswamy Layout, Bengaluru-560078 \n                                    \n \n1 \n \nModule 3 \nResearch design and Methods  \nResearch Design  \nMeaning of Research Design: The most important step after defining the research problem is  \

#### Embeddibg and VectorDB

In [11]:
import numpy as np
from typing import List, Dict,Tuple,Any
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import uuid
import chromadb
from chromadb.config import Settings

In [12]:
# Chunking (Text splitting) - splits loaded documents into smaller chunks, creates embeddings and adds to vectorstore
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Configure chunk size and overlap - adjust to your retrieval needs
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

# Ensure `documents` is available (it was created earlier by DirectoryLoader)
try:
    print(f'Preparing to split {len(documents)} document(s)')
except NameError:
    raise NameError('`documents` not found. Make sure you run the loader cell that produces `documents`.')

# Split into chunks (returns list[Document])
chunks = text_splitter.split_documents(documents)
print(f'Created {len(chunks)} chunks from {len(documents)} documents')

# Inspect a sample chunk
if len(chunks) > 0:
    print('Sample chunk (first 200 chars):')
    print(chunks[0].page_content[:200])

Preparing to split 19 document(s)
Created 125 chunks from 19 documents
Sample chunk (first 200 chars):
DAYANANDA SAGAR COLLEGE OF ENGINEERING 
         An Autonomous Institute Affiliated to VTU, Belagavi Approved by AICTE; ISO 9001:2015 
        Certified Accredited by National Assessment Accreditation


In [13]:
class EmbeddingManager:
    def __init__(self,model_name : str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()
    
    def _load_model(self):
        try:
            print(f"loading model {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully, model dimension {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {e}")
            raise

    def generate_embeddings(self,texts : List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("Model not loaded")
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape {embeddings.shape}")
        return embeddings
    
embedding_manager = EmbeddingManager()
embedding_manager
        

loading model all-MiniLM-L6-v2
Model loaded successfully, model dimension 384


#### Vector Store

In [14]:
class VectorStore:
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
    def _initialize_store(self):
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata={"description" : "PDF doc embedding for RAG"}
            )

            print(f"Vector store initialized, Collection {self.collection_name}")
            print(f"Existing documents in collection {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store {e}")
            raise
    def add_documents(self, documents : List[Any], embeddings : np.ndarray):
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match embeddings")
        print(f"Adding {len(documents)} documents to vecotor store")

        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc,embedding) in enumerate(zip(documents,embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())

        try:
            self.collection.add(
                ids = ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents = documents_text
            )
            print(f"Successfully added {len(documents)} to vector db")
            print(f"Total documents in collection {self.collection.count()}")

        except Exception as e:
            print(f"Error loading documents into vector db {e}")
            raise

vectorstore = VectorStore()
vectorstore

Vector store initialized, Collection pdf_documents
Existing documents in collection 132


In [15]:
texts = [ch.page_content for ch in chunks]
embeddings = embedding_manager.generate_embeddings(texts)
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 125 texts...


Batches: 100%|██████████| 4/4 [00:02<00:00,  1.65it/s]


Generated embeddings with shape (125, 384)
Adding 125 documents to vecotor store
Successfully added 125 to vector db
Total documents in collection 257


#### Retrieval

In [16]:
class Retriever:
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager, top_k: int = 3):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
        self.top_k = top_k

    def retrieve(self, query: str):
        """
        Takes a query string → embeds it → runs similarity search → returns retrieved documents
        """
        if not query or not isinstance(query, str):
            raise ValueError("Query must be a non-empty string.")

        # 1. Generate embedding for the query
        query_embedding = self.embedding_manager.generate_embeddings([query])[0].tolist()

        # 2. Perform similarity search in ChromaDB
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding],
                n_results=self.top_k,
                include=["documents", "metadatas", "distances"]
            )
        except Exception as e:
            print(f"Error during retrieval: {e}")
            raise

        # 3. Format response
        retrieved_docs = []
        ids = results.get("ids", [[]])[0]
        docs = results.get("documents", [[]])[0]
        metas = results.get("metadatas", [[]])[0]
        distances = results.get("distances", [[]])[0]

        for doc_id, text, meta, dist in zip(ids, docs, metas, distances):
            retrieved_docs.append({
                "id": doc_id,
                "content": text,
                "metadata": meta,
                "score": 1 - dist   # Higher = more similar
            })

        return retrieved_docs

    def pretty_print(self, retrieved_docs):
        """
        Nicely prints retrieved results with score + metadata
        """
        for i, item in enumerate(retrieved_docs, 1):
            print(f"\n📄 Result {i}")
            print(f"🔍 Score: {item['score']:.4f}")
            print(f"🆔 ID: {item['id']}")
            print(f"📌 Metadata: {item['metadata']}")
            print(f"📝 Content:\n{item['content'][:300]}...")


In [17]:
retriever = Retriever(vectorstore, embedding_manager, top_k=3)
results = retriever.retrieve("What is hypothesis testing")
# retriever.pretty_print(results)
results


Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 66.42it/s]

Generated embeddings with shape (1, 384)


[{'id': 'doc_4432a5c7_97',
  'content': 'Evidence gathered through experimental or empirical studies today is considered to be the most \npowerful support possible for a given hypothesis. \nTesting of Hypotheses: \n \nA hypothesis test is a formal way to make a decision based on statistical analysis. Hypotheses. \nHypothesis tests are tests about a population parameter. \n \nCharacteristics of hypothesis: Hypothesis must possess the following characteristics: \n \n1. Hypothesis should be clear and precise. If the hypothesis is not clear and',
  'metadata': {'title': '',
   'creationdate': '2025-07-28T16:15:39+05:30',
   'creationDate': "D:20250728161539+05'30'",
   'content_length': 496,
   'format': 'PDF 1.7',
   'moddate': '2025-07-28T16:15:39+05:30',
   'subject': '',
   'creator': 'Microsoft® Word 2016',
   'keywords': '',
   'modDate': "D:20250728161539+05'30'",
   'page': 13,
   'file_path': '../data/pdf/Module 3.pdf',
   'total_pages': 19,
   'doc_index': 97,
   'trapped': '',
 

In [20]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
from langsmith import traceable
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(groq_api_key = groq_api_key, model_name ='llama-3.1-8b-instant', temperature=0.1, max_tokens=1024)
@traceable(name = "RAG Pipeline")
def rag_simple(query,retriever,llm,top_k=3):
    results = retriever.retrieve(query)
    context = "\n\n".join([doc['content'] for doc in results])if results else ""
    if not context:
        return "no relavent context found"
    
    prompt = f"""use the following context {context}
question {query}
answer:
"""
    response = llm.invoke([prompt.format(context= context, query=query)])
    return response.content

In [21]:
prompt = input("Enter query")
answer = rag_simple(prompt,retriever,llm)
print(answer)

Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 71.48it/s]

Generated embeddings with shape (1, 384)


**What is Research Design?**

Research design is the conceptual structure within which research is conducted. It is the blueprint for the collection, measurement, and analysis of data. In essence, research design is the plan or framework that guides the researcher in collecting, analyzing, and interpreting data to answer the research question or hypothesis.

**Key Components of Research Design:**

1. **The nature of the study**: This refers to the type of research being conducted, such as descriptive, exploratory, or experimental.
2. **The purpose of the study**: This outlines the main objective of the research, including the research question or hypothesis.
3. **Decisions regarding data collection**: This includes the methods and tools used to collect data, such as surveys, interviews, or experiments.
4. **Decisions regarding data analysis**: This includes the statistical methods and techniques used to analyze the data.
5. **Decisions regarding the sample**: This includes the selectio